# Final Model Selection

`CHOICE = None` picks the model with the **lowest mean test RMSE** in `reports/results.csv`; set it to a model key
(e.g. `'random_forest'`) to choose yourself. Nothing is re-trained: the chosen model's 3-seed averaged model
(`models/<key>.joblib`) is copied to `models/final_model.joblib`, and `models/input_schema.json` is written — the
allowed choices and value ranges the app uses. Both stay on your computer (`models/` is git-ignored).

In [1]:
CHOICE = None   # None = lowest mean test RMSE; or a model key such as 'random_forest'

## 1. Setup

In [2]:
import sys, pathlib, warnings
ROOT = next((p for p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents) if (p / 'src' / 'config.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project folder (the one that contains src/).')
sys.path.insert(0, str(ROOT))   # project root -> `import src`, whatever the working folder
warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.float_format', '{:,.2f}'.format)
from src import config
from src.data_loading import load_raw
from src.models import REGISTRY, display_name, final_params, PARAM_SOURCE
from src.evaluation import read_results
from src.prediction import load_model, save_final, build_input_schema, save_input_schema

## 2. Choose the model

In [3]:
table, num = read_results()
if CHOICE is None:
    best_name = num.sort_values('Test RMSE mean')['Model'].iloc[0]
    CHOICE = next(k for k in REGISTRY if display_name(k) == best_name)
    print(f'CHOICE = {CHOICE!r} (lowest mean test RMSE)')
table[table['Model'] == display_name(CHOICE)].set_index('Model')

CHOICE = 'gradient_boosting' (lowest mean test RMSE)


,Train RMSE,CV RMSE,Val RMSE,Test RMSE,Test R²,Test MAE
Model,,,,,,
Gradient Boosting,26667.16 ± 1919.31,29049.61 ± 2435.94,30882.93 ± 9260.63,22281.94 ± 717.44,0.83 ± 0.01,9172.06 ± 436.34


## 3. Save the final model

In [4]:
final = load_model(CHOICE)
meta = {'model': display_name(CHOICE), 'key': CHOICE, 'seeds': final.seeds,
        'hyperparameters': final_params(CHOICE), 'param_source': PARAM_SOURCE[CHOICE],
        'results_mean_std_over_seeds': table[table['Model'] == display_name(CHOICE)].iloc[0].to_dict(),
        'prediction': 'mean of the per-seed pipelines, rupees per month'}
print(f'Saved {save_final(CHOICE, meta).relative_to(config.ROOT)} — {final}')

Saved models/final_model.joblib — SeedAveragedModel(Gradient Boosting, seeds=[42, 43, 44])


## 4. Save the input schema for the app

In [5]:
schema = build_input_schema(load_raw())
print(f'Saved {save_input_schema(schema).relative_to(config.ROOT)}')
for field, options in schema['choices'].items():
    print(f'  {field:<18} {options}')
print('  Localities         ' + ', '.join(f'{c}: {len(v)}' for c, v in schema['localities'].items()))
pd.DataFrame(schema['numeric']).T

Saved models/input_schema.json
  City               ['Mumbai', 'Chennai', 'Bangalore', 'Hyderabad', 'Delhi', 'Kolkata']
  Furnishing Status  ['Semi-Furnished', 'Unfurnished', 'Furnished']
  Area Type          ['Super Area', 'Carpet Area', 'Built Area']
  Tenant Preferred   ['Bachelors/Family', 'Bachelors', 'Family']
  Point of Contact   ['Contact Owner', 'Contact Agent', 'Contact Builder']
  Localities         Bangalore: 427, Chennai: 322, Delhi: 289, Hyderabad: 342, Kolkata: 252, Mumbai: 603


,min,max,median,integer
BHK,1,6,2,True
Bathroom,1,10,2,True
Size,20.00,"8,000.00",850.00,False
current_floor,-2,76,2,True
total_floors,1,89,4,True
